In [35]:
import os
import time
import random
import logging 
import numpy as np

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
import timm


In [34]:
import os

os.environ["HF_HOME"]       = "/home/kartik/.cache/huggingface"
os.environ["HF_HUB_CACHE"]  = "/home/kartik/.cache/huggingface/hub"
os.environ["HF_ASSETS_CACHE"]= "/home/kartik/.cache/huggingface/assets"
os.environ["TORCH_HOME"]    = "/home/kartik/.cache/torch"  # ← add this

In [36]:
CFG = {
    "data_root": "stratified_dataset",
    "image_size": 518,
    "batch_size": 4,
    "num_workers": 4,
    "backbone" : "vit_base_patch14_dinov2.lvd142m",
    "freeze_blocks": 8,
    "epochs" : 15,
    "lr" : 1e-4,
    "lr_backbone": 5e-5,
    "weight_decay": 1e-2,
    "seed" : 42,
    "output_dir" : "checkpoints",
    "amp" : True
}

In [37]:
def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [38]:
set_seed(CFG["seed"])

In [39]:
os.makedirs(CFG["output_dir"], exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(os.path.join(CFG["output_dir"], "train.log")),
    ],
)
log = logging.getLogger()

In [40]:
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

In [41]:
train_transform = transforms.Compose([
    transforms.Resize((CFG["image_size"]+32,CFG["image_size"]+32)),
    transforms.RandomCrop(CFG["image_size"]),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),

])

val_transform = transforms.Compose([
    transforms.Resize((CFG["image_size"],CFG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD)
])

In [42]:
root = CFG["data_root"]

In [43]:
train_dataset = datasets.ImageFolder(os.path.join(root,"train"),train_transform)
val_in_dataset = datasets.ImageFolder(os.path.join(root, "val_in_distribution"),    val_transform)
val_ood_dataset= datasets.ImageFolder(os.path.join(root, "val_out_of_distribution"),val_transform)

In [44]:
train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"],shuffle=True,num_workers=CFG["num_workers"],pin_memory=True,drop_last=True)
val_in_loader  = DataLoader(val_in_dataset,  batch_size=CFG["batch_size"]*2, shuffle=False,
                            num_workers=CFG["num_workers"], pin_memory=True)
val_ood_loader = DataLoader(val_ood_dataset, batch_size=CFG["batch_size"]*2, shuffle=False,
                            num_workers=CFG["num_workers"], pin_memory=True)

In [45]:
log.info(f"Classes : {train_dataset.classes}")   # should print ['FAKE', 'REAL']
log.info(f"Train   : {len(train_dataset):,} images")
log.info(f"Val-in  : {len(val_in_dataset):,} images")
log.info(f"Val-OOD : {len(val_ood_dataset):,} images")

2026-06-22 19:39:15,144 | Classes : ['FAKE', 'REAL']
2026-06-22 19:39:15,145 | Train   : 106,150 images
2026-06-22 19:39:15,145 | Val-in  : 10,000 images
2026-06-22 19:39:15,146 | Val-OOD : 19,168 images


In [46]:
FAKE_IDX = train_dataset.class_to_idx["FAKE"]
log.info(f"FAKE class index = {FAKE_IDX}")

2026-06-22 19:39:15,151 | FAKE class index = 0


In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"device : {device}")

2026-06-22 19:39:15,157 | device : cuda


In [49]:
backbone = timm.create_model(
    CFG["backbone"],
    pretrained=True,
    num_classes=0,
    global_pool="token",
    cache_dir="/home/kartik/.cache/torch",   # ← force it here directly
)

2026-06-22 19:40:17,832 | Loading pretrained weights from Hugging Face hub (timm/vit_base_patch14_dinov2.lvd142m)


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

2026-06-22 19:40:57,878 | [timm/vit_base_patch14_dinov2.lvd142m] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


In [50]:
for name,param in backbone.named_parameters():

    if "patch_embed" in name or "pos_embed" in name or "cls_token" in name:
        param.requires_grad = False

In [51]:
for i, block in enumerate(backbone.blocks):
    if i<CFG["freeze_blocks"]:
        for param in block.parameters():
            param.requires_grad = False

In [52]:
n_total = sum(p.numel() for p in backbone.parameters())
n_trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
log.info(f"Backbone — total params: {n_total:,} | trainable: {n_trainable:,} "
         f"({100*n_trainable/n_total:.1f}%)")

2026-06-22 19:40:57,966 | Backbone — total params: 86,579,712 | trainable: 28,359,168 (32.8%)


In [53]:
embed_dim = backbone.num_features

In [54]:
head = nn.Sequential(
    nn.Linear(embed_dim,256),
    nn.GELU(),
    nn.Dropout(0.3),
    nn.Linear(256,2),
)

In [55]:
class DINOv2Classifier(nn.Module):
    
    def __init__(self,backbone,head):
        
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self,x):
        
        features = self.backbone(x)
        return self.head(features)

In [56]:
model = DINOv2Classifier(backbone,head).to(device)

In [57]:
total_params = sum(p.numel() for p in model.parameters())
log.info(f"Full model params: {total_params:,}")

2026-06-22 19:40:58,213 | Full model params: 86,777,090


In [58]:
criterion = nn.CrossEntropyLoss()

In [59]:
backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
head_params = list(model.head.parameters())
print(len(backbone_params),len(head_params))

58 4


In [60]:
optimizer = torch.optim.AdamW([
    {"params":backbone_params,"lr":CFG["lr_backbone"]},
    {"params": head_params, "lr":CFG["lr"]},
],weight_decay=CFG["weight_decay"])

In [61]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,T_max=CFG["epochs"],eta_min=1e-6
)

In [62]:
scaler = torch.amp.GradScaler(enabled=CFG["amp"])

In [63]:
def train_one_epoch(model,loader,optimizer,criterion,scaler,device,epoch):

    model.train()
    total_loss, correct, total = 0.0,0,0

    for step, (images,labels) in enumerate(loader):

        images = images.to(device,non_blocking=True)
        labels = labels.to(device,non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type='cuda',enabled=CFG["amp"]):

            logits = model(images)
            loss = criterion(logits,labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item()

        if (step+1)%100 == 0:
            log.info(f"  Epoch {epoch} | step {step+1}/{len(loader)} | "
                     f"loss={total_loss/(step+1):.4f} | "
                     f"acc={correct/total:.4f}")
            
        
    return total_loss/len(loader),correct/total

In [64]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    
    model.eval()
    total_loss, correct, total = 0.0,0,0

    
    for images, labels in loader:

        images = images.to(device,non_blocking=True)
        labels = labels.to(device,non_blocking=True)

        with torch.amp.autocast(device_type='cuda',enabled=CFG["amp"]):
            logits = model(images)
            loss = criterion(logits,labels)

        preds = logits.argmax(dim=1)
        correct += (preds==labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item()

    return total_loss/ len(loader), correct/total

    

    


In [65]:

best_ood_acc = 0.0
log.info("\n" + "="*55)
log.info("  Starting training")
log.info("="*55)

2026-06-22 19:40:58,280 | 
2026-06-22 19:40:58,281 |   Starting training
2026-06-22 19:40:58,281 | =======================================================


In [66]:
for epoch in range(1,CFG["epochs"]+1):

    t0 = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, device, epoch

    )

    val_in_loss, val_in_acc = evaluate(model, val_in_loader, criterion, device)
    val_ood_loss, val_ood_acc = evaluate(model, val_ood_loader, criterion,device)

    scheduler.step()

    elapsed = time.time() - t0

    log.info(
        f"Epoch {epoch:02d}/{CFG['epochs']} | {elapsed:.0f}s | "
        f"train loss={train_loss:.4f} acc={train_acc:.4f} | "
        f"val-in acc={val_in_acc:.4f} | "
        f"val-OOD acc={val_ood_acc:.4f}"
    )

    if val_ood_acc > best_ood_acc:

        best_ood_acc = val_ood_acc
        ckpt_path = os.path.join(CFG["output_dir"],"best_model.pt")

        torch.save({
            "epoch": epoch,
            "model_state":model.state_dict(),
            "cfg": CFG,
            "val_ood_acc": val_ood_acc,
            "val_in_acc":val_in_acc,
        },ckpt_path)
        log.info(f"  ★ Best OOD acc so far: {best_ood_acc:.4f} — saved to {ckpt_path}")



log.info("\n" + "="*55)
log.info(f"  Training done.  Best OOD acc = {best_ood_acc:.4f}")
log.info(f"  Best model → {os.path.join(CFG['output_dir'], 'best_model.pt')}")
log.info("="*55)



2026-06-22 19:41:12,250 |   Epoch 1 | step 100/26537 | loss=0.6865 | acc=0.6725
2026-06-22 19:41:25,511 |   Epoch 1 | step 200/26537 | loss=0.6654 | acc=0.7225
2026-06-22 19:41:38,802 |   Epoch 1 | step 300/26537 | loss=0.6505 | acc=0.7550
2026-06-22 19:41:52,142 |   Epoch 1 | step 400/26537 | loss=0.6347 | acc=0.7775
2026-06-22 19:42:05,485 |   Epoch 1 | step 500/26537 | loss=0.6124 | acc=0.7970
2026-06-22 19:42:18,873 |   Epoch 1 | step 600/26537 | loss=0.6052 | acc=0.8092
2026-06-22 19:42:32,278 |   Epoch 1 | step 700/26537 | loss=0.6008 | acc=0.8150
2026-06-22 19:42:45,699 |   Epoch 1 | step 800/26537 | loss=0.5957 | acc=0.8222
2026-06-22 19:42:59,111 |   Epoch 1 | step 900/26537 | loss=0.5903 | acc=0.8281
2026-06-22 19:43:12,546 |   Epoch 1 | step 1000/26537 | loss=0.5678 | acc=0.8360
2026-06-22 19:43:25,977 |   Epoch 1 | step 1100/26537 | loss=0.5518 | acc=0.8416
2026-06-22 19:43:39,436 |   Epoch 1 | step 1200/26537 | loss=0.5452 | acc=0.8448
2026-06-22 19:43:52,891 |   Epoch 1 |